# Khmer Automatic Speech Recognition (ASR) — Complete Experiments & Training

**Course**: Deep Learning Final Project  
**Lecturer**: Mr. Soklong HIM  
**Task**: Khmer Speech-to-Text Transcription  
**Primary Metric**: Character Error Rate (CER)  

### Approaches Compared:
1. **Approach 1**: OpenAI Whisper Fine-Tuning (Seq2Seq Transformer)
2. **Approach 2**: Meta MMS-1B Khmer CTC (Acoustic CTC Model)
3. **Approach 3 (Ablation)**: Whisper-Tiny Frozen Encoder (Linear Probe / Transfer Learning)

## 1. Hardware & GPU Check
Ensure you are running on a Google Colab **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## 2. Install Required Dependencies

In [ ]:
!pip install -q torch torchaudio transformers datasets evaluate jiwer accelerate tensorboard soundfile librosa matplotlib

## 3. Clone Repository or Upload Code
Clone your GitHub repository or upload the `src/` files directly.

In [ ]:
# Option A: Clone from GitHub (Recommended)
# !git clone https://github.com/YOUR_USERNAME/khmer_asr.git
# %cd khmer_asr

# Option B: Upload src files directly
from google.colab import files
import os
if not os.path.exists('src/finetune_whisper.py') and not os.path.exists('finetune_whisper.py'):
    print('Please upload finetune_whisper.py, train_mms.py, and evaluate_and_plot.py:')
    files.upload()
    !mkdir -p src
    !mv -f *.py src/ 2>/dev/null || true

## 4. Run Approach 1: High-Accuracy Whisper Training (10 Epochs)
To achieve high accuracy and clear Khmer transcription (CER < 25%), we train for **10 epochs** using the DDD Cambodia and FLEURS datasets.

In [ ]:
# High-Accuracy 10-Epoch Run
!python src/finetune_whisper.py \
  --output-dir "./models/whisper-tiny-khmer-accurate" \
  --model-name "openai/whisper-tiny" \
  --use-fleurs-train \
  --seed 42 \
  --max-train-samples 3000 \
  --max-eval-samples 200 \
  --max-test-samples 200 \
  --num-train-epochs 10 \
  --learning-rate 1e-4 \
  --warmup-steps 200 \
  --per-device-train-batch-size 8 \
  --per-device-eval-batch-size 8 \
  --gradient-accumulation-steps 2 \
  --logging-steps 25 \
  --eval-steps 200 \
  --save-steps 200 \
  --fp16

## 5. Run Approach 2: Meta MMS-1B Khmer (CTC Acoustic Model)
Trains/evaluates the Connectionist Temporal Classification acoustic model on the identical held-out split.

In [ ]:
!python src/train_mms.py \
  --output-dir "./models/mms-khmer-ctc" \
  --model-id "facebook/mms-1b-all" \
  --target-lang "khm" \
  --seed 42 \
  --max-train-samples 1000 \
  --max-eval-samples 200 \
  --max-test-samples 200 \
  --num-train-epochs 3 \
  --learning-rate 1e-4 \
  --per-device-train-batch-size 4 \
  --per-device-eval-batch-size 4 \
  --eval-steps 200 \
  --save-steps 200 \
  --fp16

## 6. Generate Rubric Deliverables & Comparison Figures
Generates `results/learning_curves.png`, `results/metrics_comparison.png`, and `results/summary_table.md`.

In [ ]:
!python src/evaluate_and_plot.py

from IPython.display import Image, display
print('Displaying Generated Learning Curves:')
display(Image('results/learning_curves.png'))
print('Displaying Approaches Comparison:')
display(Image('results/metrics_comparison.png'))

## 7. Package and Download Trained Weights & Results
Zips the accurate model checkpoint so you can place it into your local `models/` directory.

In [ ]:
# Zip the accurate model weights and results for download
!zip -r accurate_whisper_khmer.zip models/whisper-tiny-khmer-accurate/ results/
from google.colab import files
files.download('accurate_whisper_khmer.zip')